<img src="https://raw.githubusercontent.com/ComplianceAnalytics/aml-book1/main/assets/cal_logo_banner.png" alt="Compliance Analytics Ltd" width="300" onerror="this.style.display='none'">

# Applied AML Analytics: Turning Data Science Skills into Compliance Decisions
## Chapter 7 — Risk and Coverage Assessment
### Companion Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_07.ipynb)

---

**Book:** *Applied AML Analytics: Turning Data Science Skills into Compliance Decisions* — Book 1  
**Publisher companion repository:** [github.com/ComplianceAnalytics/aml-book1](https://github.com/ComplianceAnalytics/aml-book1)  
**Dataset:** Northgate Retail Bank (synthetic — all data is fictional)  
**Chapters covered:** 3 · 4 · 5 · 6 · 7 · **7 (this notebook)**

> **How to use this notebook**  
> Run cells top-to-bottom using **Shift+Enter** or the ▶ button. The setup cell (Section 0) must run first — it generates the Northgate dataset that all later cells depend on. You do not need to install anything; all required libraries are pre-installed in Google Colab.

---

## Contents

| Section | Description | Exercise link |
|---------|-------------|---------------|
| **0. Setup** | Generate the Northgate dataset | — |
| **1. Colab Preview** | Rule NRB-GEO-003 (high-risk country counterparty) · three-rule coverage matrix (mirrors Section 7.7 of the text) | — |
| **2. Exercise 7.1 Extension** | Country breakdown · FFIEC red-flag mapping · coverage gap identification | Exercise 7.1 |
| **3. Reflection cells** | Structured answer prompts | Exercise 7.1 Parts A–C |

---
## Section 0 — Setup: Generate the Northgate Dataset

**Run this cell first.** It generates four CSV files in the Colab session's working directory:

| File | Rows | Description |
|------|------|-------------|
| `nb_transactions.csv` | ~23,000 | All account transactions, Jan–Dec 2023 |
| `nb_customers.csv` | 500 | Customer and account records |
| `nb_counterparties.csv` | 300 | Counterparty firms and their country codes |
| `nb_accounts.csv` | 500 | Account metadata |

The dataset is **fully synthetic**. Northgate Retail Bank does not exist. All account IDs, names, and transactions are generated from a fixed random seed for educational purposes only.

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

rng = np.random.default_rng(42)

# ── Counterparties ────────────────────────────────────────────────────────────
HIGH_RISK  = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']
LOW_RISK   = ['US', 'GB', 'DE', 'FR', 'CA', 'AU', 'SG', 'JP', 'NL', 'CH']

n_cpty      = 300
cpty_ids    = [f'CPT{i:04d}' for i in range(1, n_cpty + 1)]
cpty_cc     = (rng.choice(HIGH_RISK, size=30).tolist() +
               rng.choice(LOW_RISK,  size=270, replace=True).tolist())
rng.shuffle(cpty_cc)
df_cpty = pd.DataFrame({'counterparty_id': cpty_ids, 'country_code': cpty_cc})
df_cpty.to_csv('nb_counterparties.csv', index=False)

n_cust   = 500
cust_ids = [f'NRB_{i:03d}' for i in range(1, n_cust + 1)]
acct_ids = [f'ACC{i:04d}' for i in range(1, n_cust + 1)]
mule_idx = list(range(6))

occupations = ['Employed', 'Self-Employed', 'Retired', 'Student', None]
occ_probs   = [0.55, 0.20, 0.12, 0.08, 0.05]
crr_scores  = rng.choice([1,2,3,4,5], p=[0.35,0.30,0.20,0.10,0.05], size=n_cust)
for i in mule_idx:
    crr_scores[i] = rng.choice([3,4])
incomes_k = rng.lognormal(mean=3.1, sigma=0.5, size=n_cust) * 1000
for i in mule_idx:
    incomes_k[i] = rng.uniform(18, 24) * 1000

df_cust = pd.DataFrame({
    'customer_id':       cust_ids,
    'account_id':        acct_ids,
    'occupation':        rng.choice(occupations, p=occ_probs, size=n_cust),
    'crr_score':         crr_scores,
    'stated_income_usd': np.round(incomes_k, -2),
    'account_open_date': [
        (date(2020,1,1) + timedelta(days=int(d))).isoformat()
        for d in rng.integers(0, 1460, size=n_cust)
    ],
})
df_cust.to_csv('nb_customers.csv', index=False)
df_cust.to_csv('nb_accounts.csv',  index=False)

txn_rows = []
start    = date(2023, 1, 1)
txn_id   = 1

for i, (cid, aid) in enumerate(zip(cust_ids, acct_ids)):
    is_mule = i in mule_idx
    if is_mule:
        for m in range(12):
            for _ in range(rng.integers(3, 9)):
                day      = rng.integers(1, 28)
                txn_date = date(2023, m + 1, day)
                amount   = round(rng.uniform(7800, 9800), 2)
                cpty     = rng.choice(cpty_ids[:30])
                txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                                 'txn_date': txn_date.isoformat(), 'txn_type': 'CASH_IN',
                                 'amount': amount, 'counterparty_id': cpty})
                txn_id += 1
    else:
        for _ in range(rng.integers(12, 80)):
            txn_date = start + timedelta(days=int(rng.integers(0, 365)))
            txn_type = rng.choice(['CASH_IN','TRANSFER_OUT','TRANSFER_IN','CARD'],
                                   p=[0.15, 0.35, 0.35, 0.15])
            amount   = round(min(rng.lognormal(6.5, 1.2), 50000), 2)
            cpty_pool = cpty_ids[30:] if rng.random() > 0.03 else cpty_ids[:30]
            txn_rows.append({'txn_id': f'TXN{txn_id:06d}', 'account_id': aid,
                             'txn_date': txn_date.isoformat(), 'txn_type': txn_type,
                             'amount': amount, 'counterparty_id': rng.choice(cpty_pool)})
            txn_id += 1

df_txn = (pd.DataFrame(txn_rows)
            .assign(txn_date=lambda d: pd.to_datetime(d['txn_date']))
            .sort_values('txn_date')
            .reset_index(drop=True))
df_txn.to_csv('nb_transactions.csv', index=False)

print(f"✅ Dataset generated")
print(f"   nb_counterparties : {len(df_cpty):>6,} rows")
print(f"   nb_customers      : {len(df_cust):>6,} rows")
print(f"   nb_transactions   : {len(df_txn):>6,} rows")
print(f"   Date range        : {df_txn['txn_date'].min().date()} → {df_txn['txn_date'].max().date()}")

---
## Section 1 — Colab Preview: Rule NRB-GEO-003 and the Coverage Matrix

> *This section mirrors Section 7.7 of the textbook exactly. Run the cells and compare the output to the printed figures.*

### The complete three-rule system

| Rule ID | Detection logic | Threshold |
|---------|-----------------|-----------|
| NRB-STRUCT-001 | Rolling 30-day cash deposits | > USD 7,500, ≥ 3 txns |
| NRB-VEL-002 | Rapid-fire velocity | ≥ 5 txns in 14 days |
| **NRB-GEO-003** | High-risk country counterparty | ≥ 2 txns, ≥ USD 5,000 total |

**NRB-GEO-003** targets a different dimension of risk: the geographic profile of an account's counterparties. The mule accounts in the Northgate dataset were designed to transact primarily with counterparties in high-risk jurisdictions (KP, IR, MM, SY, YE, AF, LY). This rule should therefore capture all six mule accounts.

### Coverage assessment methodology

A coverage assessment asks: *for each known money-laundering typology, do we have at least one rule that would detect it?* The FFIEC Bank Secrecy Act/AML Examination Manual lists red-flag indicators. A well-designed TM system should have measurable coverage across these indicators.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_txn  = pd.read_csv('nb_transactions.csv', parse_dates=['txn_date'])
df_cpty = pd.read_csv('nb_counterparties.csv')
MULE_IDS = [f'ACC{i:04d}' for i in range(1, 7)]
HIGH_RISK_COUNTRIES = ['KP', 'IR', 'MM', 'SY', 'YE', 'AF', 'LY']

def apply_rule_1(df_txn, threshold=7500, min_txns=3):
    cash = df_txn[(df_txn['txn_type']=='CASH_IN') & (df_txn['amount']<10_000)].copy()
    cash = cash.sort_values(['account_id','txn_date'])
    results = []
    for acct, grp in cash.groupby('account_id'):
        grp = grp.set_index('txn_date').sort_index()
        if grp['amount'].rolling('30D').sum().max() > threshold and grp['amount'].rolling('30D').count().max() >= min_txns:
            results.append({'account_id': acct})
    return pd.DataFrame(results)

def apply_rule_2(df_txn, min_txns=5, window_days=14, max_gap_days=3):
    df = df_txn.copy().sort_values(['account_id','txn_date'])
    results = []
    for acct, grp in df.groupby('account_id'):
        grp = grp.set_index('txn_date').sort_index()
        rc = grp['amount'].rolling(f'{window_days}D').count()
        if rc.max() >= min_txns:
            gaps = grp.index.to_series().diff().dt.days.dropna()
            if (gaps <= max_gap_days).sum() >= (min_txns - 1):
                results.append({'account_id': acct})
    return pd.DataFrame(results)

def apply_rule_3(df_txn, df_cpty, high_risk_countries, min_txns=2, min_amount=5000):
    """Rule NRB-GEO-003: transactions with high-risk country counterparties."""
    hr_cpty = df_cpty[df_cpty['country_code'].isin(high_risk_countries)]['counterparty_id']
    hr_txns = df_txn[df_txn['counterparty_id'].isin(hr_cpty)].copy()
    results = []
    for acct, grp in hr_txns.groupby('account_id'):
        total = grp['amount'].sum()
        countries = df_cpty.loc[df_cpty['counterparty_id'].isin(grp['counterparty_id']),
                                'country_code'].unique().tolist()
        if len(grp) >= min_txns and total >= min_amount:
            results.append({
                'account_id':      acct,
                'hr_txn_count':    len(grp),
                'total_hr_amount': round(total, 2),
                'countries':       ', '.join(sorted(countries)),
            })
    return pd.DataFrame(results)

alerts_r1 = apply_rule_1(df_txn)
alerts_r2 = apply_rule_2(df_txn)
alerts_r3 = apply_rule_3(df_txn, df_cpty, HIGH_RISK_COUNTRIES)

print(f"Rule 1 alerts: {len(alerts_r1)}")
print(f"Rule 2 alerts: {len(alerts_r2)}")
print(f"Rule 3 alerts: {len(alerts_r3)}")
print()
print("Rule 3 — top 10 by total high-risk amount:")
print(alerts_r3.sort_values('total_hr_amount', ascending=False).head(10).to_string(index=False))

In [ ]:
# Three-rule coverage matrix for the 6 mule accounts
coverage = []
for m in MULE_IDS:
    coverage.append({
        'account_id': m,
        'Rule 1 (Structuring)':    '✓' if m in set(alerts_r1['account_id']) else '✗',
        'Rule 2 (Velocity)':       '✓' if m in set(alerts_r2['account_id']) else '✗',
        'Rule 3 (Geo Risk)':       '✓' if m in set(alerts_r3['account_id']) else '✗',
    })

coverage_df = pd.DataFrame(coverage).set_index('account_id')
print("Three-Rule Coverage Matrix — Known Mule Accounts:")
print(coverage_df.to_string())
print()
# Rules hit per account
coverage_df['rules_triggered'] = (
    (coverage_df['Rule 1 (Structuring)'] == '✓').astype(int) +
    (coverage_df['Rule 2 (Velocity)'] == '✓').astype(int) +
    (coverage_df['Rule 3 (Geo Risk)'] == '✓').astype(int)
)
print("Rules triggered per mule account:")
print(coverage_df['rules_triggered'].to_string())

In [ ]:
# Country breakdown chart for high-risk transactions
hr_cpty_ids = df_cpty[df_cpty['country_code'].isin(HIGH_RISK_COUNTRIES)]['counterparty_id']
hr_txns = df_txn[df_txn['counterparty_id'].isin(hr_cpty_ids)]
merged  = hr_txns.merge(df_cpty, on='counterparty_id')
country_counts = merged.groupby('country_code').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
bars = country_counts.plot(kind='bar', ax=ax, color='#4472C4', width=0.6)
ax.set_title('High-Risk Country Transactions — Northgate Dataset', fontsize=11, fontweight='bold', pad=10)
ax.set_xlabel('Country Code', fontsize=10)
ax.set_ylabel('Number of Transactions', fontsize=10)
ax.tick_params(axis='x', rotation=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

**What you're seeing:** The bar chart shows the distribution of high-risk country transactions across the seven listed jurisdictions. This is the raw material for a geographic risk assessment: if one country code dominates the dataset, the bank's exposure to that jurisdiction warrants additional scrutiny.

The coverage matrix shows how many of the three rules each mule account triggers. An account triggering all three rules is exhibiting structuring behaviour, velocity anomalies, AND geographic risk — three independent red flags pointing at the same account.

---
## Section 2 — Exercise 7.1 Extension: Coverage Gap Analysis

> *The main-text exercise asks you to build the three-rule coverage matrix. This extension asks you to identify what typologies the three rules do NOT cover, and what a fourth rule might target.*

In [ ]:
# Full alert population: which accounts are never caught by any rule?
all_alerted = set(alerts_r1['account_id']) | set(alerts_r2['account_id']) | set(alerts_r3['account_id'])
print(f"Total unique accounts alerted by any rule: {len(all_alerted)}")
print(f"Total accounts in dataset: 500")
print(f"Accounts with NO alert: {500 - len(all_alerted)}")
print()

# FFIEC-style coverage mapping
ffiec_typologies = [
    ('Structuring (sub-threshold cash deposits)',       True,  False, False),
    ('Rapid succession of deposits/withdrawals',        False, True,  False),
    ('Transactions with high-risk jurisdictions',       False, False, True),
    ('Unusual cash activity relative to account type',  True,  False, False),
    ('Round-dollar transaction patterns',               False, False, False),
    ('Fan-out/fan-in funnel patterns',                  False, False, False),
    ('Dormant account sudden activation',               False, False, False),
    ('Peer-group behavioural outlier',                  False, False, False),
]

print("FFIEC Red-Flag Coverage Mapping:")
print(f"{'Typology':<50} {'R1':>4} {'R2':>4} {'R3':>4} {'Covered':>8}")
print("-" * 70)
for typ, r1, r2, r3 in ffiec_typologies:
    covered = '✓' if (r1 or r2 or r3) else '✗ GAP'
    r1s = '✓' if r1 else ' '
    r2s = '✓' if r2 else ' '
    r3s = '✓' if r3 else ' '
    print(f"{typ:<50} {r1s:>4} {r2s:>4} {r3s:>4} {covered:>8}")

**✏️ YOUR OBSERVATION**

Look at the coverage table. Which typologies are NOT covered by any of the three rules? 

For each gap, propose in one sentence what a fourth rule might look like (what transaction pattern it would detect, and what threshold parameter it would use).

*Write your answer in Section 3, Part C.*

---
## Section 3 — Reflection: Exercise 7.1 Answer Cells

> *Double-click any cell to edit it.*

#### Part A — Rule 3 Performance

*(Edit this cell to write your answer)*

**How many accounts triggered Rule NRB-GEO-003 at default settings?**  
  
**Are all six mule accounts captured? If not, which are missing and why might that be?**  
  
**Which high-risk country code appears most frequently in the Northgate dataset?**  


#### Part B — Three-Rule Coverage Matrix

*(Edit this cell to write your answer)*

**How many mule accounts are caught by all three rules simultaneously?**  
  
**Are there any mule accounts caught by only one rule? Which rule catches them, and what does this suggest about their behaviour?**  
  
**In a real-world TM system, what would you do with accounts that trigger three rules vs. accounts that trigger only one?**  


#### Part C — Coverage Gaps

*(Edit this cell to write your answer)*

**List two FFIEC typologies that the three-rule Northgate system does NOT cover.**  
  
**For each gap, describe in one sentence what a fourth rule might look like.**  
  
**How would you present coverage gaps to a compliance committee? What evidence would you bring?**  


---
## What's Next

In Chapter 8, you will apply **Isolation Forest** — an unsupervised machine learning algorithm — to score all alerted accounts by anomaly level. The three rule flags (R1, R2, R3) become features, alongside behavioural metrics. All six mule accounts should rank in the top positions by anomaly score, providing an ML-assisted prioritisation layer on top of the rule-based system.

Open Chapter 8: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_08.ipynb)